# Analisis & Pengolahan Data Kos Putri SCBD (Seturan, Sleman)
Notebook ini berisi alur lengkap data preparation, cleaning, feature engineering, dan ekspor data bersih untuk **Sistem 1: Interactive Web Dashboard & Catalog**.

In [72]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Load Data
Baris pertama pada file CSV asli berupa koma kosong (`,,,,,,,`), sehingga kita gunakan `header=1` agar nama kolom terbaca dengan benar.

In [73]:
df = pd.read_csv("KOST PUTRI SCBD - Kost Putri.csv", header=1)
df.head()

,No.,Nama Kos,Nama Pemilik Kos,Biaya Perbulan/semester,Biaya Tambahan,Kontak Kos,Alamat,Fasilitas Kos,Link Google Maps,Nama Rekomendasi,Jurusan Rekomendasi,Kontak
0,1,Kost Putri Asri,Ibu Novi,Rp750.000-1.200.000/bulan\n(sesuai tipe kamar),-,wa.me//628985892539,"Puluhdadi No.416, Ngropoh, Caturtunggal, \nKec...","Kasur, kamar mandi dalam, wifi-internet, listr...",Kos Putri Asri,NaN,NaN,NaN
1,2,Kost Putri Pak Seno,-,800.000 (kipas angin)-\n1.000.000 (ac)/bulan,menginap lebih \ndari 2 hari \nRp25.000,wa.me//6285228622686,"Jl. Seturan No.1, Kledokan, Caturtunggal, \nKe...","kamar mandi luar 4, wifi, listrik, kasur, \nle...",Kos Putri Pak Seno,NaN,NaN,NaN
2,3,Kost Perum APH \nSeturan A.24,-,RP600.000-700.000/bulan\n(sesuai tipe kamar),-,wa.me//6281313199951,"Perum APH Seturan Blok A.24 \nKledokan, Caturt...","kamar mandi luar 2, listrik, kasur, lemari pak...",Kledokan,NaN,NaN,NaN
3,4,Graha Seturan\nKost Putri,-,Rp850.000,NaN,wa.me//6285799247890,"Jln. Seturan Raya, Kledokan, Caturtunggal, \nD...","kasur, meja kecil, lemari pakaian, \nkamar man...",https://maps.app.goo.gl/279if6Z1PG6ZnCK97,NaN,NaN,NaN
4,5,Kost Diajeng,Ibu Sri Sawarsih,Rp900.000/bulan\ndibayar pertigabulan,NaN,wa.me//6285382754735,"Puluhdadi, Kledokan, Caturtunggal, \nKec. Depo...","kasur, lemari, wifi, kamar mandi dalam. \nlist...",Kost Putri Muslim Diajeng,NaN,NaN,NaN


## 2. Karakteristik & Profiling Data Awal

In [74]:
# Informasi struktur dataset
df.info()

print("\n--- Jumlah Missing Values per Kolom ---")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63 entries, 0 to 62
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   No.                      63 non-null     int64  
 1   Nama Kos                 61 non-null     object 
 2   Nama Pemilik Kos         11 non-null     object 
 3   Biaya Perbulan/semester  30 non-null     object 
 4   Biaya Tambahan           5 non-null      object 
 5   Kontak Kos               63 non-null     object 
 6   Alamat                   53 non-null     object 
 7   Fasilitas Kos            29 non-null     object 
 8   Link Google Maps         52 non-null     object 
 9   Nama Rekomendasi         0 non-null      float64
 10  Jurusan Rekomendasi      0 non-null      float64
 11  Kontak                   0 non-null      float64
dtypes: float64(3), int64(1), object(8)
memory usage: 6.0+ KB

--- Jumlah Missing Values per Kolom ---
No.                         0
Nama Kos

## 3. Data Cleaning & Preprocessing
- Mempertahankan seluruh 63 data kos (mengisi Nama Kos kosong dengan penomoran unik)
- Menghapus kolom yang 100% kosong (`Nama Rekomendasi`, `Jurusan Rekomendasi`, `Kontak`)
- Membersihkan karakter baris baru (`\n`) pada teks

In [75]:
df_clean = df.copy()

# 1. Beri nama jelas untuk kos yang namanya belum terdata
df_clean['Nama Kos'] = df_clean['Nama Kos'].fillna(
    df_clean['No.'].apply(lambda x: f"Kos Putri (No. {x})")
)

# 2. Hapus kolom 100% kosong
drop_cols = ['Nama Rekomendasi', 'Jurusan Rekomendasi', 'Kontak']
df_clean.drop(columns=[col for col in drop_cols if col in df_clean.columns], inplace=True)

# 3. Bersihkan spasi dan newline berlebih
df_clean['Nama Kos'] = df_clean['Nama Kos'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
df_clean['Alamat'] = df_clean['Alamat'].fillna('Alamat belum tercatat').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
df_clean['Fasilitas Kos'] = df_clean['Fasilitas Kos'].fillna('Fasilitas belum tercatat / konfirmasi pemilik').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

print(f"Total data setelah cleaning awal: {len(df_clean)} baris")

Total data setelah cleaning awal: 63 baris


## 4. Feature Engineering

### A. Ekstraksi Harga Numerik (`harga_min` dan `harga_max`)
Mengubah teks seperti `Rp750.000-1.200.000/bulan` menjadi angka numerik integer.

In [76]:
def extract_price(text):
    if pd.isna(text) or str(text).strip() in ['-', '']:
        return np.nan, np.nan
    text_clean = str(text).replace('.', '').replace(',', '')
    # Ambil semua angka dengan minimal 6 digit
    numbers = [int(n) for n in re.findall(r'\d{6,}', text_clean)]
    
    if not numbers:
        return np.nan, np.nan
    elif len(numbers) == 1:
        return numbers[0], numbers[0]
    else:
        return min(numbers), max(numbers)

prices = df_clean['Biaya Perbulan/semester'].apply(extract_price)
df_clean['harga_min'] = [p[0] for p in prices]
df_clean['harga_max'] = [p[1] for p in prices]

df_clean[['Nama Kos', 'Biaya Perbulan/semester', 'harga_min', 'harga_max']].head()

,Nama Kos,Biaya Perbulan/semester,harga_min,harga_max
0,Kost Putri Asri,Rp750.000-1.200.000/bulan\n(sesuai tipe kamar),750000.0,1200000.0
1,Kost Putri Pak Seno,800.000 (kipas angin)-\n1.000.000 (ac)/bulan,800000.0,1000000.0
2,Kost Perum APH Seturan A.24,RP600.000-700.000/bulan\n(sesuai tipe kamar),600000.0,700000.0
3,Graha Seturan Kost Putri,Rp850.000,850000.0,850000.0
4,Kost Diajeng,Rp900.000/bulan\ndibayar pertigabulan,900000.0,900000.0


### B. Deteksi Fasilitas Cerdas (Boolean Flags)
Menggabungkan teks dari kolom `Fasilitas Kos` dan `Biaya Perbulan/semester` agar fasilitas yang ditulis pada deskripsi harga (seperti AC) tetap terdeteksi dengan tepat.

In [77]:
gabungan_info = (
    df_clean['Fasilitas Kos'].fillna('').astype(str) + " " + 
    df_clean['Biaya Perbulan/semester'].fillna('').astype(str)
).str.lower()

df_clean['has_ac'] = gabungan_info.str.contains(r'\bac\b|air conditioner', regex=True)
df_clean['has_km_dalam'] = gabungan_info.str.contains(r'kamar mandi dalam|km dalam|km\. dalam', regex=True)
df_clean['has_wifi'] = gabungan_info.str.contains(r'wifi|internet', regex=True)
df_clean['has_kasur'] = gabungan_info.str.contains(r'kasur', regex=True)
df_clean['has_listrik'] = gabungan_info.str.contains(r'listrik', regex=True)

# Cek frekuensi masing-masing fasilitas
print("Jumlah kos per fasilitas:")
print(df_clean[['has_ac', 'has_km_dalam', 'has_wifi', 'has_kasur', 'has_listrik']].sum())

Jumlah kos per fasilitas:
has_ac           7
has_km_dalam    17
has_wifi        18
has_kasur       21
has_listrik     15
dtype: int64


### C. Standarisasi URL WhatsApp & Google Maps

In [78]:
# Standarisasi WhatsApp ke link resmi https://wa.me/62...
def format_wa(link):
    if pd.isna(link) or str(link).strip() in ['-', '']:
        return None
    raw = str(link).strip()
    digits = re.sub(r'\D', '', raw)
    if not digits:
        return None
    if digits.startswith('08'):
        digits = '628' + digits[2:]
    elif digits.startswith('8'):
        digits = '628' + digits[1:]
    return f"https://wa.me/{digits}"

df_clean['wa_url'] = df_clean['Kontak Kos'].apply(format_wa)

# Standarisasi Google Maps URL
def format_maps(link):
    if pd.isna(link) or str(link).strip() in ['-', '']:
        return None
    link = str(link).strip()
    if link.startswith('http'):
        return link
    return f"https://www.google.com/maps/search/?api=1&query={link.replace(' ', '+')}"

df_clean['maps_url'] = df_clean['Link Google Maps'].apply(format_maps)

df_clean[['Nama Kos', 'wa_url', 'maps_url']].head()

,Nama Kos,wa_url,maps_url
0,Kost Putri Asri,https://wa.me/628985892539,https://www.google.com/maps/search/?api=1&quer...
1,Kost Putri Pak Seno,https://wa.me/6285228622686,https://www.google.com/maps/search/?api=1&quer...
2,Kost Perum APH Seturan A.24,https://wa.me/6281313199951,https://www.google.com/maps/search/?api=1&quer...
3,Graha Seturan Kost Putri,https://wa.me/6285799247890,https://maps.app.goo.gl/279if6Z1PG6ZnCK97
4,Kost Diajeng,https://wa.me/6285382754735,https://www.google.com/maps/search/?api=1&quer...


## 5. Simpan Data Bersih (Export)
Dataset ini yang akan menjadi sumber data untuk aplikasi Streamlit (`app.py`).

In [79]:
# Simpan di root dan folder data/
df_clean.to_csv("kost_putri_cleaned.csv", index=False)
df_clean.to_csv("data/kost_putri_cleaned.csv", index=False)

print(f"Dataset bersih berhasil disimpan! Total kos: {len(df_clean)} baris.")

Dataset bersih berhasil disimpan! Total kos: 63 baris.


## 6. Ringkasan Eksplorasi Data (Quick EDA)
Melihat gambaran distribusi harga kos di daerah Seturan / SCBD.

In [80]:
valid_prices = df_clean['harga_min'].dropna()

print("=== Statistik Harga Kos per Bulan ===")
print(f"Harga Termurah : Rp {int(valid_prices.min()):,}")
print(f"Rata-rata      : Rp {int(valid_prices.mean()):,}")
print(f"Median         : Rp {int(valid_prices.median()):,}")
print(f"Harga Termahal : Rp {int(valid_prices.max()):,}")

=== Statistik Harga Kos per Bulan ===
Harga Termurah : Rp 450,000
Rata-rata      : Rp 1,018,166
Median         : Rp 850,000
Harga Termahal : Rp 3,300,000
